# Sound classification with Freesound descriptors and an SVM

**Sound and Music Description — guided Python example**

This notebook illustrates a complete **supervised sound-classification pipeline**. It retrieves precomputed audio descriptors from [Freesound](https://freesound.org/), forms a fixed-length feature vector for each recording, and trains a **support vector machine (SVM)** to distinguish short examples of **violin**, **naobo** (Chinese cymbals), and **trumpet**. A new sound (by default, a viola) illustrates how an already trained classifier makes predictions.

The example progresses from **audio descriptors → labeled dataset → training/test split → feature scaling → SVM → evaluation → a new prediction**. Code cells produce plots and outputs that illustrate each step. Each stage is a worked demonstration with an explanation of the concepts.

### Concepts illustrated

- **Descriptors and feature vectors:** how heterogeneous audio characteristics become numerical model inputs.
- **Supervised learning:** the role of class labels, examples, and a train/test split.
- **Standardization and data leakage:** why scaling must be fitted on the training set only.
- **SVM classification and evaluation:** nonlinear boundaries, accuracy, precision, recall, F1, and a confusion matrix.
- **Generalization:** how to interpret predictions for a sound outside the original categories, and why automatically collected labels may be noisy.

**Environment:** basic Python and NumPy. Internet access and a personal Freesound API key are needed when downloading sounds for the first time; cached files can be reused. The notebook preserves the original **train/test → SVM → query → export** structure and the result bundle used by the companion visualization notebook.

**Freesound API version:** this revision uses the descriptor names documented for the current APIv2, rather than the older `lowlevel.*` feature names. The extracted features are computed by Freesound and retrieved through its API; **Essentia is not run locally** in this notebook.

## 1. Setup and required packages

We use **requests** to call the Freesound REST API, **NumPy** for numerical arrays, **scikit-learn** for the classifier and evaluation, and **Matplotlib** for plots. No Essentia installation is necessary here because Freesound already supplies its extracted audio descriptors.

Install the packages in the Python environment used by this notebook, if necessary:

```python
%pip install numpy requests scikit-learn matplotlib python-dotenv
```

If you installed packages while the kernel was running, restart it and run the notebook from the top.

In [ ]:
import json
import os
import pickle
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests
from dotenv import load_dotenv
from IPython.display import Audio, display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print('Packages imported successfully.')

## 2. Authenticate with your own Freesound API key

Freesound requires an **API key** to identify applications that use its API. To obtain one:

1. Create an account or sign in at [freesound.org](https://freesound.org/).
2. Visit [Freesound API credentials](https://freesound.org/apiv2/apply/), request credentials, and copy the **Client secret/Api key** value (not the client ID).
3. The next cell checks for a key. If `FREESOUND_API_KEY` is already an environment variable or is set in a local `.env` file, it will be used automatically. Otherwise, **type or paste your key into the hidden prompt**.

The entered key is held **only in this running Python process**, via `os.environ`; it is not saved in the notebook, output, or any of the result files. Do not put an API key directly in a code cell, upload it to GitHub, or share a `.env` file. **Never print the key.**

Optional `.env` file (stored beside the notebook, not committed to version control):

```text
FREESOUND_API_KEY=your_personal_api_key
```

Reference: [Freesound authentication documentation](https://freesound.org/docs/api/authentication.html).

In [ ]:
# 1. Read a key already present in the environment or a private .env file.
load_dotenv(override=False)
api_key = os.environ.get('FREESOUND_API_KEY', '').strip()

# 2. Otherwise, allow the notebook user to enter their key (input is hidden).
if not api_key:
    api_key = getpass('Enter your Freesound API key (hidden): ').strip()
    if not api_key:
        raise RuntimeError('An API key is required to download new sounds.')
    os.environ['FREESOUND_API_KEY'] = api_key

print('An API key is available. It has not been displayed or written to disk.')

## 3. Audio descriptors: from sound to feature vectors

A recording has many samples (a waveform), but an SVM expects a **fixed-length vector** of numbers. Freesound supplies descriptors that summarize aspects of the sound's spectrum and temporal envelope.

For each recording, we use the following descriptors:

| Descriptor | What it measures | Typical interpretation |
|---|---|---|
| **Spectral centroid** | Weighted average frequency of the spectrum | Higher often corresponds to a brighter sound |
| **Spectral energy** | Energy of spectral content | Related to signal strength, though strongly affected by recording level |
| **Dissonance** | Sensory roughness associated with interacting spectral peaks | Potentially helpful for distinguishing harmonic and inharmonic sounds |
| **Pitch salience** | Strength of tonal periodicity | A high value often indicates a clear pitched sound |
| **Inharmonicity** | Departure of partial frequencies from an ideal harmonic series | May distinguish cymbal-like and harmonic instrument sounds |
| **Log attack time** | Logarithm of the duration of the sound's attack | Relates to sharp versus gradual onsets |
| **MFCC** | Coefficients describing the broad spectral envelope on a mel scale | Often useful for timbre classification |
| **HPCP** | Energy in pitch classes, averaged over time | Describes tonal content; more useful for pitch/harmony than pure instrument identity |

For example, Freesound's **MFCC** descriptor is a vector of **13 values**, and **HPCP** has **36 values** (three bins per semitone). We can choose individual components, such as `mfcc.1`, as separate input variables. **The number of feature dimensions is not the number of sounds.** If we have 60 recordings and 12 selected features, the data matrix is shape `(60, 12)`.

### A useful mathematical example: spectral centroid

For a magnitude spectrum $ |X[k]| $ and bin frequencies $f_k$,

$$
C = \frac{\sum_k f_k |X[k]|}{\sum_k |X[k]|}.
$$

The actual values retrieved below are **summary descriptors calculated by Freesound's analysis pipeline**, not a frame-by-frame recomputation by this notebook.

**Feature design matters.** Features with different units and ranges need scaling before training; MFCCs and HPCP bins are not directly comparable without it. Some descriptors may be unavailable for some sounds, and those records should be reported and excluded rather than silently replaced with zeros.

In [ ]:
# Descriptor names supported by the current Freesound analysis API.
SCALAR_DESCRIPTORS = [
    'spectral_centroid', 'spectral_energy', 'dissonance',
    'pitch_salience', 'inharmonicity', 'log_attack_time',
]
VECTOR_DESCRIPTORS = {'mfcc': 13, 'hpcp': 36}
API_DESCRIPTORS = SCALAR_DESCRIPTORS + list(VECTOR_DESCRIPTORS)

# Each entry names ONE numeric feature (one column of the SVM input matrix).
feature_names = SCALAR_DESCRIPTORS + [
    f'{descriptor}.{i}'
    for descriptor, count in VECTOR_DESCRIPTORS.items()
    for i in range(count)
]

# Fixed feature selection for this example; the same order is used throughout the pipeline.
# MFCC 0 is omitted by default to emphasize spectral-envelope shape rather than total level.
selected_feature_names = [
    'spectral_centroid', 'spectral_energy', 'dissonance',
    'pitch_salience', 'inharmonicity', 'log_attack_time',
    'mfcc.1', 'mfcc.2', 'mfcc.3', 'mfcc.4', 'mfcc.5', 'mfcc.6',
]

assert selected_feature_names
assert len(set(selected_feature_names)) == len(selected_feature_names)
assert all(name in feature_names for name in selected_feature_names)
print(f'Using {len(selected_feature_names)} features:', ', '.join(selected_feature_names))

## 4. Download the training sounds (first run only)

We create a small training dataset by searching Freesound with text and optional tags. For this illustration, each search expression is *treated as a provisional class label*:

- **violin**, tag **pizzicato**
- **naobo**, without an extra tag
- **trumpet**, tag **single-note**

The original setup requests **20 recordings per class**, with durations from **0 to 3 seconds**. Search results come from user-contributed tags and descriptions: they are **not automatically verified ground truth**. Listening to a sample and correcting labels is part of responsible dataset construction.

For each selected sound we save its **numeric analysis** (`features.json`), **metadata** (`metadata.json`), and optionally a **low-quality MP3 preview**. We also preserve the link, creator, and license for attribution. The files will be cached so subsequent runs do not need to download them again.

The code uses the current `GET /apiv2/search/` and `GET /apiv2/sounds/{id}/analysis/` endpoints and authenticates with a header. It never uses `eval()` on data returned by a remote API.

In [ ]:
# ===== EXAMPLE CONFIGURATION =====
training_dir = Path('svm_train_sounds')  # New format; do not mix with older cache files.
training_queries = [
    ('violin', 'pizzicato', 20),
    ('naobo', '', 20),
    ('trumpet', 'single-note', 20),
]
training_duration = (0, 3)  # seconds
save_previews = True

# Test and model parameters
test_size = 0.20
random_state = 42
kernel = 'rbf'
C = 10.0
gamma = 'scale'

# Query to classify after training
query_sound_name = 'viola'
query_tag = 'pizzicato'
query_duration = (0, 3)
query_top_n_results = 1
query_dir = Path('svm_query_sound')
results_file = Path('svm_results/svm_core_results.pkl')
# =================================

### 4.1 How the API and cache helpers work

The next code cell contains the infrastructure for the example. The functions make each data-processing step explicit:

1. `make_session()` sends the Freesound key in an **Authorization header**.
2. `api_get_json()` performs a request and decodes the returned JSON.
3. `download_class_sounds()` retrieves the search results and caches sound analyses, metadata, and optional previews. Sounds without the required descriptors are skipped.
4. `load_dataset()` converts descriptor arrays to finite feature vectors and constructs a data matrix and label vector.

Caching means the data can be inspected repeatedly without downloading every file again. The loader also removes **duplicate Freesound IDs** appearing under different classes to reduce train/test leakage. As with any search-based dataset, text queries and tags are imperfect proxies for verified labels.

In [ ]:
BASE_URL = 'https://freesound.org/apiv2'


def make_session(api_key):
    session = requests.Session()
    session.headers.update({'Authorization': f'Token {api_key}'})
    # Retry occasional server/rate-limit failures, respecting Retry-After.
    retries = Retry(total=3, backoff_factor=0.5,
                    status_forcelist=[429, 500, 502, 503, 504],
                    allowed_methods=['GET'], respect_retry_after_header=True)
    session.mount('https://', HTTPAdapter(max_retries=retries))
    return session


def api_get_json(session, endpoint, *, params=None):
    url = endpoint if endpoint.startswith('https://freesound.org/apiv2/') else BASE_URL + endpoint
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


def feature_vector(analysis, names):
    """Convert current APIv2 descriptor JSON to a selected 1D numerical vector."""
    values = []
    for name in names:
        if '.' in name:  # For example, mfcc.3
            descriptor, index = name.rsplit('.', 1)
            vector = analysis.get(descriptor)
            if not isinstance(vector, (list, tuple)) or len(vector) <= int(index):
                raise ValueError(f'Missing/short descriptor: {descriptor}')
            value = vector[int(index)]
        else:
            value = analysis.get(name)
        if value is None or isinstance(value, (list, tuple, dict, bool)):
            raise ValueError(f'Missing or non-scalar descriptor: {name}')
        value = float(value)
        if not np.isfinite(value):
            raise ValueError(f'Non-finite descriptor: {name}')
        values.append(value)
    return np.asarray(values, dtype=float)


def download_preview(preview_url, target):
    """Previews may be hosted externally: do NOT send the API token to that URL."""
    if not preview_url:
        return
    with requests.get(preview_url, stream=True, timeout=30) as response:
        response.raise_for_status()
        with target.open('wb') as handle:
            for chunk in response.iter_content(chunk_size=64 * 1024):
                if chunk:
                    handle.write(chunk)


def download_class_sounds(session, query, tag, count, target_root, duration,
                          save_previews=True, max_candidates=200, cache_folder=None):
    """Cache at most `count` valid sounds per class, without deleting existing files."""
    class_dir = target_root / (cache_folder or query)
    class_dir.mkdir(parents=True, exist_ok=True)
    existing_ids = {
        d.name for d in class_dir.iterdir()
        if d.is_dir() and (d / 'features.json').exists()
    }
    if len(existing_ids) >= count:
        print(f'{query}: {len(existing_ids)} cached sounds; no download needed.')
        return

    filters = [f'duration:[{duration[0]} TO {duration[1]}]']
    if tag:
        filters.insert(0, f'tag:{tag}')

    page = 1
    seen = 0
    added = 0
    skipped = 0
    while len(existing_ids) + added < count and seen < max_candidates:
        payload = api_get_json(session, '/search/', params={
            'query': query,
            'filter': ' '.join(filters),
            'sort': 'score',
            'fields': 'id,name,url,username,license,previews',
            'page_size': 30,
            'page': page,
        })
        results = payload.get('results', [])
        if not results:
            break
        for sound in results:
            if len(existing_ids) + added >= count or seen >= max_candidates:
                break
            seen += 1
            sound_id = str(sound['id'])
            if sound_id in existing_ids:
                continue
            sound_dir = class_dir / sound_id
            feature_file = sound_dir / 'features.json'
            if feature_file.exists():
                continue
            try:
                analysis = api_get_json(session, f'/sounds/{sound_id}/analysis/',
                                        params={'fields': ','.join(API_DESCRIPTORS)})
                # We only require descriptors selected for this run.
                feature_vector(analysis, selected_feature_names)
            except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
                skipped += 1
                if skipped <= 3:
                    print(f'  Skipping {sound_id}: {type(exc).__name__}: {exc}')
                continue

            sound_dir.mkdir(parents=True, exist_ok=True)
            with feature_file.open('w', encoding='utf-8') as f:
                json.dump(analysis, f, indent=2)
            with (sound_dir / 'metadata.json').open('w', encoding='utf-8') as f:
                json.dump({k: sound.get(k) for k in
                           ['id', 'name', 'url', 'username', 'license']}, f, indent=2)
            if save_previews:
                preview = (sound.get('previews') or {}).get('preview-lq-mp3')
                try:
                    download_preview(preview, sound_dir / 'preview.mp3')
                except requests.RequestException as exc:
                    print(f'  Preview unavailable for {sound_id}: {type(exc).__name__}')
            added += 1
        if not payload.get('next'):
            break
        page += 1
    print(f'{query}: {len(existing_ids)} cached + {added} new; {skipped} incomplete/failed.')


def load_dataset(root, selected_names, class_filter=None):
    features, labels, ids, paths = [], [], [], []
    seen_ids = set()
    # Only files named features.json from this API version.
    for file_path in sorted(root.glob('*/*/features.json')):
        if class_filter and file_path.parent.parent.name != class_filter:
            continue
        if file_path.parent.name in seen_ids:
            print(f'Skipping duplicate Freesound ID {file_path.parent.name} to prevent data leakage.')
            continue
        try:
            analysis = json.loads(file_path.read_text(encoding='utf-8'))
            vector = feature_vector(analysis, selected_names)
        except (ValueError, TypeError, KeyError, json.JSONDecodeError) as exc:
            print(f'Skipping {file_path}: {exc}')
            continue
        features.append(vector)
        labels.append(file_path.parent.parent.name)
        ids.append(file_path.parent.name)
        paths.append(file_path)
        seen_ids.add(file_path.parent.name)
    if not features:
        return np.empty((0, len(selected_names))), np.array([]), [], []
    return np.vstack(features), np.asarray(labels), ids, paths

### 4.2 Build and inspect the dataset

The matrix $X$ has one row per sound and one column per selected descriptor. The vector $y$ contains one **class label** per sound. We keep the Freesound IDs separately so we can identify the original recordings later.

Only the **first run** needs to contact Freesound for missing data. Re-running this cell uses existing cached files and downloads only what is missing. Changing `selected_feature_names` affects which features are read from the cached JSON; if a cached sound lacks a newly selected descriptor, it is skipped.

In [ ]:
session = make_session(os.environ['FREESOUND_API_KEY'])

try:
    for class_name, class_tag, target_count in training_queries:
        download_class_sounds(
            session, query=class_name, tag=class_tag, count=target_count,
            target_root=training_dir, duration=training_duration,
            save_previews=save_previews,
        )
except requests.HTTPError as exc:
    raise RuntimeError('Freesound request failed: check your key, API permissions and connectivity.') from exc

full_features, full_labels, sound_ids, feature_paths = load_dataset(
    training_dir, selected_feature_names
)
if len(full_features) == 0:
    raise RuntimeError('No usable descriptor files found. Review the search and API key.')

class_names, class_counts = np.unique(full_labels, return_counts=True)
print('\nDataset shape (sounds × features):', full_features.shape)
print('Class distribution:', dict(zip(class_names, class_counts)))
print('Feature order:', selected_feature_names)
if len(class_names) < 2 or np.any(class_counts < 2):
    raise RuntimeError('Need at least two classes and two usable sounds per class.')

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(class_names, class_counts)
ax.set(title='Number of sounds per class', ylabel='Recordings')
plt.show()

### 4.3 Visualize the descriptor space and listen to the sound classes

The scatter plot below uses **spectral centroid** and **inharmonicity** as two example dimensions. Different point clouds may suggest that these descriptors contain information useful for classification. Overlap in this 2D projection is compatible with separability in the full feature space used by the SVM.

The same cell displays one cached recording from each class and its source information. Listening makes the relationship between **acoustic properties**, **descriptors**, and **search-derived labels** more concrete. If a label does not match the audible content, it illustrates a limitation of building training sets from unverified search metadata.

In [ ]:
centroid_index = selected_feature_names.index('spectral_centroid')
inharmonicity_index = selected_feature_names.index('inharmonicity')

fig, ax = plt.subplots(figsize=(7, 4))
for label in class_names:
    mask = full_labels == label
    ax.scatter(full_features[mask, centroid_index],
               full_features[mask, inharmonicity_index],
               alpha=0.75, label=label)
ax.set(xlabel='Spectral centroid', ylabel='Inharmonicity',
       title='Two audio descriptors for the training examples')
ax.legend()
plt.show()

for label in class_names:
    example_path = next(path for path, y in zip(feature_paths, full_labels) if y == label)
    preview = example_path.parent / 'preview.mp3'
    meta_path = example_path.parent / 'metadata.json'
    meta = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
    print(f'{label}: {meta.get("name", example_path.parent.name)} '
          f'— by {meta.get("username", "unknown")}, license: {meta.get("license", "unknown")}')
    print('Source:', meta.get('url', 'unknown'))
    if preview.exists():
        display(Audio(filename=str(preview)))
    else:
        print('No cached preview (use the source link).')

## 5. Split the data **before** standardization

We use **80% of the sounds for training** and reserve **20% for testing**. `stratify=full_labels` attempts to preserve approximately the same class proportions in both sets. A fixed `random_state` makes the random split reproducible.

**Why not scale the whole dataset first?** The mean and standard deviation would then contain information from the test set, a form of **data leakage**. Instead, a scikit-learn **pipeline** learns scaling parameters *only from training data* and applies exactly the same transform to the test and query sounds.

For feature $j$, standardization uses training statistics:

$$
z_{i,j}=\frac{x_{i,j}-\mu_{j,\mathrm{train}}}{\sigma_{j,\mathrm{train}}}.
$$

The test data must remain unseen until evaluation. Note that repeated recordings from the same artist, performance, or recording session may still be correlated; a more demanding evaluation would split by recording source or uploader.

In [ ]:
train_indices, test_indices = train_test_split(
    np.arange(len(full_features)),
    test_size=test_size,
    random_state=random_state,
    stratify=full_labels,
)
train_features = full_features[train_indices]
test_features = full_features[test_indices]
train_labels = full_labels[train_indices]
test_labels = full_labels[test_indices]

print('Training X:', train_features.shape, '| labels:', train_labels.shape)
print('Test X:    ', test_features.shape, '| labels:', test_labels.shape)
print('Held-out Freesound IDs:', [sound_ids[i] for i in test_indices[:5]])

## 6. Train the Support Vector Machine (SVM)

An **SVM** seeks a decision boundary that separates classes. With `kernel='rbf'`, it can produce **nonlinear** boundaries in feature space. The controls are:

- **`C`**: penalty for misclassification. Larger values prioritize fitting training examples, potentially increasing overfitting.
- **`gamma`**: how locally each example influences the RBF boundary. `gamma='scale'` selects a data-dependent default.
- **`probability=True`**: enables class-probability estimates using an additional calibration procedure. On a small dataset, these values should **not be treated as reliable confidence guarantees**.

The classifier is **supervised**: training samples are paired with labels. The result depends strongly on the correctness of the tags and on the selected features.

In [ ]:
model = make_pipeline(
    StandardScaler(),
    SVC(kernel=kernel, C=C, gamma=gamma,
        probability=True, random_state=random_state),
)
model.fit(train_features, train_labels)
print(f'Trained {kernel.upper()} SVM with C={C}, gamma={gamma}.')

# We also keep the scaled matrices for the separate visualization notebook.
scaler = model.named_steps['standardscaler']
svm = model.named_steps['svc']
scaled_train_features = scaler.transform(train_features)
scaled_test_features = scaler.transform(test_features)

## 7. Evaluate the classifier on new (held-out) sounds

The **confusion matrix** compares true test labels with predictions. Row $i$ corresponds to the actual class; column $j$ is the predicted class. Values on the diagonal are correct classifications.

- **Accuracy:** fraction of sounds correctly classified.
- **Precision:** when the model predicts a class, how often is it correct?
- **Recall:** of the sounds belonging to a class, how many are correctly retrieved?
- **F1-score:** harmonic mean of precision and recall.

The weighted scores below average across classes using the number of samples in each class. Compare **training vs. test accuracy**: a large gap can suggest overfitting. A high test score with only a handful of recordings is **not conclusive evidence** of generalization.

In [ ]:
train_pred = model.predict(train_features)
test_pred = model.predict(test_features)

print(f'Training accuracy: {accuracy_score(train_labels, train_pred):.1%}')
print(f'Test accuracy:     {accuracy_score(test_labels, test_pred):.1%}')
print(f'Test precision:    {precision_score(test_labels, test_pred, average="weighted", zero_division=0):.1%}')
print(f'Test recall:       {recall_score(test_labels, test_pred, average="weighted", zero_division=0):.1%}')
print(f'Test F1:           {f1_score(test_labels, test_pred, average="weighted", zero_division=0):.1%}')
print('\nPer-class evaluation:\n')
print(classification_report(test_labels, test_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(test_labels, test_pred,
                                        labels=svm.classes_, ax=ax,
                                        cmap='Blues', colorbar=False)
ax.set_title('SVM confusion matrix — held-out test recordings')
plt.show()

## 8. Predict the class of a new sound

The following code searches for a new example (default: **viola**, with tag `pizzicato`), retrieve its descriptors, and apply the **same trained pipeline**. The query's search term is **not a verified true label**. A viola was deliberately chosen because it is acoustically related to the training violin class but is not one of the trained categories.

If the model predicts `violin`, this means **“closest to the violin class according to the selected descriptors and this model”**, not that Freesound or an expert has identified the sound as a violin. The SVM cannot predict a fourth class that it has not been trained on.

We also show the sound source and license, and play its cached MP3 preview when available. Changing the query **term, tag, or duration** creates a separate query cache so old results are not mistakenly reused.

In [ ]:
# Cache query results under a key containing the query and filter.
# This avoids reusing an earlier query when the example query term or tag changes.
query_cache_key = f'{query_sound_name}__{query_tag or "any"}__{query_duration[0]}-{query_duration[1]}'

# Query retrieval: use the same helper, features and search restrictions.
try:
    download_class_sounds(
        session, query=query_sound_name, tag=query_tag,
        count=query_top_n_results, target_root=query_dir,
        duration=query_duration, save_previews=True, cache_folder=query_cache_key,
    )
except requests.HTTPError as exc:
    raise RuntimeError('Could not retrieve the query sound from Freesound.') from exc

query_matrix, query_names, query_ids, query_paths = load_dataset(
    query_dir, selected_feature_names, class_filter=query_cache_key
)
if len(query_matrix) == 0:
    raise RuntimeError('No query sound available. Try another query or tag.')

query_index = 0  # Use the first usable retrieved sound as the query example.
query_features = query_matrix[query_index]
query_file = query_paths[query_index]
query_meta_file = query_file.parent / 'metadata.json'
query_meta = json.loads(query_meta_file.read_text(encoding='utf-8')) if query_meta_file.exists() else {}

predicted_label = model.predict(query_features.reshape(1, -1))[0]
probabilities = model.predict_proba(query_features.reshape(1, -1))[0]
predicted_probability = float(probabilities[list(svm.classes_).index(predicted_label)])
scaled_query_features = scaler.transform(query_features.reshape(1, -1))[0]

print(f'Search term: {query_sound_name!r} (NOT verified ground truth)')
print(f'Freesound sound ID: {query_ids[query_index]}')
print('Sound:', query_meta.get('name', '[metadata unavailable]'))
print('Creator:', query_meta.get('username', '[unknown]'))
print('Source:', query_meta.get('url', '[unknown]'))
print('License:', query_meta.get('license', '[unknown]'))
print(f'Predicted class: {predicted_label} ({predicted_probability:.1%} estimated probability)')

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(svm.classes_, probabilities)
ax.set(title='SVM class-probability estimates for the query',
       ylabel='Estimated probability', ylim=(0, 1))
plt.show()

preview_path = query_file.parent / 'preview.mp3'
if preview_path.exists():
    display(Audio(filename=str(preview_path)))
else:
    print('No preview cached. Use the Freesound source URL to listen.')

## 9. Export results for the separate visualization notebook

The original notebook saved a pickle bundle with scaled training/test features, support-vector indices, a query prediction, and SVM parameters. We **retain these original keys** so an existing visualization notebook can continue to read them. This revision also includes the new `selected_feature_names`, a class list, and Freesound source metadata.

**Safety:** pickle files can execute code when loaded. Only open pickle files you generated yourself or obtained from a trusted source. Never include the Freesound API key in the exported bundle.

In [ ]:
combined_scaled_features = np.vstack((scaled_train_features, scaled_test_features))
combined_labels = np.concatenate((train_labels, test_labels))
combined_sound_ids = [sound_ids[i] for i in train_indices] + [sound_ids[i] for i in test_indices]
test_mask = np.concatenate((
    np.zeros(len(train_features), dtype=bool),
    np.ones(len(test_features), dtype=bool),
))

bundle = {
    # Original visualization-notebook keys:
    'combined_scaled_features': combined_scaled_features,
    'combined_labels': combined_labels,
    'combined_sound_ids': combined_sound_ids,
    'test_mask': test_mask,
    'support_indices': svm.support_,  # Relative to the TRAINING subset, which comes first above.
    'scaled_query_features': scaled_query_features,
    'query_label': query_sound_name,  # Search term, NOT verified true label.
    'predicted_label': predicted_label,
    'predicted_probability': predicted_probability,
    'kernel': kernel,
    'C': C,
    'gamma': gamma,
    # Useful extra metadata:
    'selected_feature_names': selected_feature_names,
    'class_names': svm.classes_.tolist(),
    'query_sound_id': query_ids[query_index],
    'query_source_url': query_meta.get('url'),
    'query_license': query_meta.get('license'),
    'query_is_ground_truth': False,
}

results_file.parent.mkdir(parents=True, exist_ok=True)
with results_file.open('wb') as f:
    pickle.dump(bundle, f)
print(f'Saved results to: {results_file.resolve()}')
print('Saved keys:', ', '.join(bundle))

## 10. Worked comparison: how the feature representation affects classification

The preceding SVM uses a combination of **scalar audio descriptors** and **MFCCs**. A classifier can produce different boundaries depending on which features describe the sounds. To make this effect visible, the next cell automatically compares three representations:

- **Scalar descriptors:** spectral centroid, energy, dissonance, pitch salience, inharmonicity, and attack time.
- **MFCC coefficients:** a compact representation of the spectral envelope.
- **Combined representation:** the descriptors used in the main classifier.

Each representation is evaluated using **stratified cross-validation on the original training data**, with a fresh `StandardScaler` and SVM inside each fold. The held-out test set from section 7 is **not used** in this comparison. Cross-validation estimates will vary with the downloaded recordings, labels, and small sample size; the plots illustrate how the choice of representation matters, rather than identifying a universally best feature set.

This comparison also demonstrates a useful distinction: features describing **signal level**, **spectral timbre**, and **pitch content** are not interchangeable, and classification performance depends on which differences between classes are acoustically relevant.

In [ ]:
# Compare representations using cross-validation on TRAINING data only.
# This is a worked illustration: the held-out test data remain untouched.
feature_groups = {
    'Scalar descriptors': [n for n in selected_feature_names if n in SCALAR_DESCRIPTORS],
    'MFCC coefficients': [n for n in selected_feature_names if n.startswith('mfcc.')],
    'Combined (main model)': list(selected_feature_names),
}
feature_groups = {label: names for label, names in feature_groups.items() if names}

min_train_per_class = int(np.min(np.unique(train_labels, return_counts=True)[1]))
n_folds = min(3, min_train_per_class)
if n_folds < 2:
    print('Not enough training recordings per class for a cross-validation comparison.')
else:
    folds = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    cv_results = {}
    for group_name, names in feature_groups.items():
        indices = [selected_feature_names.index(name) for name in names]
        X_subset = train_features[:, indices]
        # Each CV fold independently learns its own scaling statistics.
        model_for_group = make_pipeline(
            StandardScaler(), SVC(kernel=kernel, C=C, gamma=gamma)
        )
        scores = cross_val_score(model_for_group, X_subset, train_labels,
                                 cv=folds, scoring='accuracy')
        cv_results[group_name] = scores
        print(f'{group_name:<24} {len(names):>2} features: '
              f'mean accuracy {scores.mean():.1%}, std {scores.std():.1%}')

    names = list(cv_results)
    means = [cv_results[name].mean() for name in names]
    stds = [cv_results[name].std() for name in names]
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.bar(names, means, yerr=stds, capsize=5)
    ax.set(ylim=(0, 1.05), ylabel='CV accuracy (mean ± std)',
           title=f'Illustration: feature representations ({n_folds}-fold training CV)')
    plt.xticks(rotation=10, ha='right')
    plt.tight_layout()
    plt.show()


## 11. Interpretation and references

This example connects a sound description to a prediction in several stages. **Freesound descriptors** replace a variable-length waveform with a fixed-length numerical vector. **Standardization** puts heterogeneous measurements onto comparable scales. The **SVM** learns boundaries between the three search-derived classes. The **confusion matrix** shows where its test predictions agree or disagree with those provisional labels; the final **query prediction** demonstrates that a closed-set classifier must choose one of the classes it has seen, even for a sound that might belong to none of them.

The feature comparison in section 10 highlights why **the choice of descriptors matters**. MFCCs primarily encode broad spectral-envelope shape, whereas scalar measures such as inharmonicity and pitch salience describe other properties. Accuracy values should be interpreted cautiously: the dataset is small, Freesound tags are not verified labels, and recordings may share artists or recording conditions. Scores from the example are **illustrative, not a benchmark**.

The notebook also preserves metadata and license references. Freesound previews and sound files should be used according to their individual licenses, with attribution where required. The exported pickle bundle is intended only for the companion visualization example and should be loaded only from a trusted source.

**References:** [Freesound API overview](https://freesound.org/docs/api/overview.html), [audio descriptors](https://freesound.org/docs/api/analysis_docs.html), [SVM documentation](https://scikit-learn.org/stable/modules/svm.html), [scikit-learn cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html), [data leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html).